# Week 3 · Diagnose and Improve Training

**Core activity 2:** Complete paper activity 1 first, then work through the optimizer experiments through RMSProp. **Optional activities 3–4** explore minibatch noise and generalization; return to worksheet activity 5 afterwards.

Work in pairs to allow for some discussion. Predict → change a setting → inspect → explain. Derivatives are supplied explicitly. Run cells in order using the repository's Python environment. All cells run as supplied: an unfinished warm-up prints a reminder without blocking later activities. Later activities do not depend on solving the warm-up.

Data, models, and plots are supplied. Change the clearly marked controls, not the data-generation seed or split during a comparison. Write answers in the **Your record** cells. For environment setup help, see `README.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
    "text.color": "#202020", "axes.labelcolor": "#202020",
    "xtick.color": "#202020", "ytick.color": "#202020", "font.size": 10})
DTYPE = torch.float64
print("Ready: CPU calculations; NumPy", np.__version__, "PyTorch", torch.__version__)


## 2. Repair the optimizer

### 2A. Complete one update
We reuse $L(u,v)=\tfrac12(u^2+10v^2)$. Replace `None` in `student_step` by a gradient-descent expression using `theta`, `grad`, and `lr`. If you leave it unfinished, the cell prints a reminder and you can continue with the supplied experiments. The check uses a point different from the paper example.

This small function is for SGD practice; the later momentum comparison has its own function for you to implement.

In [ ]:
def sgd_step(theta, grad, lr):
    # candidate = None  # TODO: replace None with the gradient-descent update.    
    return candidate

probe = np.array([2., -1.])
probe_gradient = np.array([2., -10.])
candidate = sgd_step(probe, probe_gradient, .05)
if candidate is None:
    print("Warm-up not completed yet. Replace None when ready; you may continue to the experiments.")
else:
    np.testing.assert_allclose(candidate, [1.9, -.5])
    print("Your update passed the check.")


In [ ]:
def sgd_run(lr, steps=40):
    theta = np.array([1., 1.])
    path = [theta.copy()]
    for _ in range(steps):
        grad = theta * np.array([1., 10.])
        theta = sgd_step(theta, grad, lr)
        path.append(theta.copy())
        if not np.isfinite(theta).all() or np.linalg.norm(theta) > 1e6:
            break  # Keep an unstable experiment from overflowing.
    path = np.array(path)
    losses = .5 * (path[:, 0]**2 + 10 * path[:, 1]**2)
    return path, losses


def momentum_run(lr, beta, steps=40):
    theta = np.array([1., 1.])
    velocity = np.zeros(2)
    path = [theta.copy()]
    for _ in range(steps):
        grad = theta * np.array([1., 10.])
        theta, velocity = momentum_step(theta, grad, velocity, lr, beta)
        path.append(theta.copy())
        if not np.isfinite(theta).all() or np.linalg.norm(theta) > 1e6:
            break  # Keep an unstable experiment from overflowing.
    path = np.array(path)
    losses = .5 * (path[:, 0]**2 + 10 * path[:, 1]**2)
    return path, losses


def plot_quadratic(runs):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    u, v = np.meshgrid(np.linspace(-1.5, 1.5, 160), np.linspace(-1.5, 1.5, 160))
    axes[0].contour(u, v, .5 * (u*u + 10*v*v), levels=[.05,.2,.5,1,2,4,8,12], colors="0.75")
    for name, (path, losses) in runs.items():
        axes[0].plot(path[:,0], path[:,1], '.-', markersize=3, label=name)
        axes[1].semilogy(np.arange(len(losses)), np.maximum(losses, 1e-16), label=name)
    axes[0].plot(0, 0, 'k*', markersize=10)
    axes[0].set(xlabel="u", ylabel="v", xlim=(-1.5,1.5), ylim=(-1.5,1.5), title="Parameter paths (fixed viewing window)")
    axes[1].set(xlabel="Updates", ylabel="Loss", title="Loss includes the initial value")
    axes[1].legend(fontsize=8)
    fig.tight_layout(); plt.show()


### 2B. Test your learning-rate predictions
Before running, copy your one-step results for 0.01, 0.1, and 0.25 from the worksheet and predict what further updates will do. Inspect both the contour plot and loss curves. A divergent path can leave the fixed contour window; it has not disappeared from training.

1. Which predictions were supported? Identify slow progress, a coordinate reaching zero immediately, and divergence.
2. Add learning rate 0.15 and inspect the path. Does crossing from one side of zero to the other necessarily imply increasing loss?
3. Compare all runs from the same starting point and with the same update budget. Explain why these controls matter.

In [ ]:
# CHANGE THESE CONTROLS, then rerun this cell.
learning_rates = [.01, .1, .25]
steps = 40
plain_runs = {f"SGD: lr={lr}": sgd_run(lr, steps=steps) for lr in learning_rates}
plot_quadratic(plain_runs)


**Your record — 2B:** predictions / observations / explanation of alternating coordinates versus loss.

_Write here._


### Background: momentum and adaptive optimization

Gradient descent updates parameters with $\theta_{t+1}=\theta_t-\eta g_t$, where $g_t=\nabla L(\theta_t)$ and one learning rate $\eta$ scales every coordinate. If the loss is steep in one direction and flat in another, a large step can bounce across the steep direction while a small step makes slow progress along the flat one.

**Momentum** keeps a decaying memory of recent gradients. In this notebook’s convention, $m_{t+1}=\beta m_t+g_t$ and $\theta_{t+1}=\theta_t-\eta m_{t+1}$. Gradients that keep pointing in a similar direction build up; changes in direction partly cancel. This can smooth zigzags and build speed along a valley. The stored velocity can also carry a parameter over a hill, as in activity 2D. Too much momentum can overshoot, and momentum starting from rest cannot move from a point where the gradient is exactly zero.

**RMSProp** adapts each parameter’s step using a decaying average of its squared gradients: $s_t=\rho s_{t-1}+(1-\rho)g_t^2$. It scales each step by $g_t/(\sqrt{s_t}+\epsilon)$. Recent gradients matter more and older scale history fades, which can help when gradient magnitudes change during training. A small $\epsilon$ prevents division by zero.

Momentum and RMSProp change the path across the loss surface, not the objective, and neither guarantees finding a global minimum. Activity 2C examines momentum history; optional 2D shows how it can cross a local minimum; 2E compares RMSProp with momentum when directions have different scales.

### 2C. Add gradient history
Write the parameter vector as $\theta=(u,v)$. Use $m_t$ for the momentum accumulator, so it is distinct from the coordinate $v$. The supplied rule is $m_{t+1}=\beta m_t+\nabla L(\theta_t)$, $\theta_{t+1}=\theta_t-\eta m_{t+1}$, with $m_0=0$. The landscape remains fixed; momentum changes the path across it.

Before running, use $m_{t+1}=\beta m_t+g_t$ with $m_0=0$, $\beta=0.5$, and gradients $[2,2,-1]$ to calculate the three momentum values. After the gradient changes sign, what direction is the next parameter update, and why?

Implement `student_momentum_step` in the next cell. It should return both the updated parameters and updated momentum. The first update matches SGD because the initial momentum is zero; the difference appears after gradient history accumulates.

Predict and compare SGD at `lr=0.15`, momentum at `(lr=0.15, beta=0.4)`, and the stress setting `(lr=0.4, beta=0.9)`. The last changes **two** settings, so it is not an isolated test of momentum.

1. Compare the final loss and first update below 0.01 for each setting. If the threshold is never reached, record that.
2. Use the paths to explain when gradient history helps or hurts. Does reaching the threshold guarantee the loss stays below it?

We compare updates for this tiny example, not wall-clock performance or universal optimizer rankings.

In [ ]:
def momentum_step(theta, grad, velocity, lr, beta):
    new_velocity = None  # TODO: combine the old velocity and current gradient.
    new_theta = None     # TODO: update theta using the new velocity.
    return new_theta, new_velocity

probe_theta = np.array([2., -1.])
probe_gradient = np.array([2., -10.])
probe_velocity = np.zeros(2)
first_theta, first_velocity = momentum_step(
    probe_theta, probe_gradient, probe_velocity, lr=.05, beta=.4
)
if first_theta is None or first_velocity is None:
    momentum_step_completed = False
    print("Complete the two TODOs before running the momentum comparison.")
else:
    np.testing.assert_allclose(first_velocity, [2., -10.])
    np.testing.assert_allclose(first_theta, [1.9, -.5])
    second_theta, second_velocity = momentum_step(
        first_theta, np.array([1., -5.]), first_velocity, lr=.05, beta=.4
    )
    np.testing.assert_allclose(second_velocity, [1.8, -9.])
    np.testing.assert_allclose(second_theta, [1.81, -.05])
    momentum_step_completed = True
    print("Momentum update passed both checks.")


In [ ]:
# Compare SGD with momentum. Keep the same initial point and number of updates.
if not momentum_step_completed:
    print("Finish the momentum-step implementation above, then rerun this cell.")
else:
    momentum_runs = {
        "SGD: lr=0.15": sgd_run(0.15, steps=40),
        "Momentum: lr=0.15, beta=0.4": momentum_run(0.15, 0.4, steps=40),
        "Momentum stress: lr=0.4, beta=0.9": momentum_run(0.4, 0.9, steps=40),
    }
    plot_quadratic(momentum_runs)
    for name, (path, losses) in momentum_runs.items():
        hit = np.flatnonzero(losses < .01)
        print(name, "final loss:", f"{losses[-1]:.3e}",
              "first update below .01:", int(hit[0]) if len(hit) else "not reached")


**Your record — 2C:** settings / first update to threshold / final loss / explanation / limitation of the comparison.

_Write here._


### Momentum across local minima

Consider the one-dimensional loss $L(x)=3(x^2-1)^2-1.5x$. It has a higher local minimum on the left and a much lower global minimum on the right, separated by a steep hill. Starting from $x_0=-1.6$, compare gradient descent with several momentum values.

1. Predict which settings cross the hill, then compare the paths and final losses. Which run overshoots, and how does the accumulated direction explain it?
2. What happens if you start exactly at the left minimum with zero momentum?

All runs use the same start, learning rate, and update budget; only momentum changes. These results illustrate this particular loss and do not guarantee escape on other landscapes.

In [ ]:
def double_well_loss(x):
    return 3 * (x**2 - 1)**2 - 1.5 * x

def double_well_grad(x):
    return 12 * x * (x**2 - 1) - 1.5

def double_well_run(lr, beta, steps=150, start=-1.6):
    x, velocity = start, 0.0
    path = [x]
    for _ in range(steps):
        velocity = beta * velocity + double_well_grad(x)
        x = x - lr * velocity
        path.append(x)
    return np.array(path)

# Change only beta; compare paths from the same starting point.
learning_rate = 0.001
momentum_values = [0.0, 0.5, 0.8, 0.9, 0.95]
minima_runs = {f"beta={beta:g}": double_well_run(lr=learning_rate, beta=beta)
               for beta in momentum_values}
colors = plt.cm.tab10(np.arange(len(minima_runs)))
x_grid = np.linspace(-1.8, 1.8, 500)
ncols = min(3, len(minima_runs))
nrows = (len(minima_runs) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.4 * nrows),
                       sharex=True, sharey=True, squeeze=False)
axes = axes.ravel()
markers = ["o", "s", "^", "D", "v"]
for i, (ax, (label, path)) in enumerate(zip(axes, minima_runs.items())):
    ax.plot(x_grid, double_well_loss(x_grid), color="0.78", linewidth=1.5, zorder=1)
    ax.plot(path, double_well_loss(path), color=colors[i], linewidth=2.2, zorder=3)
    ax.plot(path[::10], double_well_loss(path[::10]), linestyle="none",
            marker=markers[i % len(markers)], color=colors[i], markersize=6,
            markeredgecolor="black", markeredgewidth=0.6, zorder=4)
    ax.scatter([path[0]], [double_well_loss(path[0])], marker="*", s=150,
               color="black", edgecolor="white", linewidth=0.8, zorder=6)
    ax.scatter([path[-1]], [double_well_loss(path[-1])], marker="o", s=65,
               color=colors[i], edgecolor="black", linewidth=0.8, zorder=6)
    ax.set_title(f"{label}  |  final loss={double_well_loss(path[-1]):.2f}")
    ax.set_xlim(-1.8, 1.8); ax.set_ylim(-2.1, 10.2)
    ax.grid(alpha=0.15)
    print(f"{label}: final x={path[-1]:.3f}, final loss={double_well_loss(path[-1]):.3f}")
for ax in axes[len(minima_runs):]:
    ax.set_visible(False)
fig.supxlabel("x", y=0.035); fig.supylabel("L(x)")
fig.suptitle("Each panel shows the full trajectory on the same loss surface")
fig.text(0.5, 0.01, "Gray: loss surface   Colored line: every update   Colored markers: every 10 updates   Black star: shared start   Circle: final point", ha="center", fontsize=9)
fig.tight_layout(rect=(0, 0.08, 1, 0.94)); plt.show()

**Your record — 2D:** predictions / which beta values crossed / final positions and losses / why momentum helps here but does not guarantee escape.

_Write here._

### 2E. RMSProp across differently scaled directions

Momentum can build speed in a consistent direction, but it still applies one shared learning rate to every coordinate. On $L(u,v)=\frac12(u^2+100v^2)$, the loss is much steeper in $v$ than in $u$. RMSProp can scale the two coordinates differently.

1. At $(u,v)=(1,1)$, the gradient is $(1,100)$. With $\rho=0.99$ and $s_0=(0,0)$, calculate $s_1$ and the first RMSProp update at learning rate $0.03$ (ignore $\epsilon$). Why are the coordinate updates similar despite the 100-fold gradient difference?
2. Compare the supplied momentum and RMSProp settings over 100 updates. Predict which reaches loss below $0.1$ sooner, then use the contour and loss plots to explain when coordinate-wise scaling helps. These settings illustrate one case, not a universal ranking.

In [ ]:
scaled_curvature = np.array([1.0, 100.0])
def scaled_loss(theta):
    return 0.5 * np.sum(scaled_curvature * theta**2)

def scaled_momentum_run(lr=0.0009, beta=0.9, steps=100):
    theta = np.ones(2)
    momentum = np.zeros(2)
    losses = [scaled_loss(theta)]
    path = [theta.copy()]
    for _ in range(steps):
        gradient = scaled_curvature * theta
        momentum = beta * momentum + gradient
        theta = theta - lr * momentum
        losses.append(scaled_loss(theta))
        path.append(theta.copy())
    return np.array(losses), np.array(path)

def scaled_rmsprop_run(lr=0.03, rho=0.99, steps=100):
    theta = np.ones(2)
    square_average = np.zeros(2)
    losses = [scaled_loss(theta)]
    path = [theta.copy()]
    for _ in range(steps):
        gradient = scaled_curvature * theta
        square_average = rho * square_average + (1 - rho) * gradient**2
        theta = theta - lr * gradient / (np.sqrt(square_average) + 1e-8)
        losses.append(scaled_loss(theta))
        path.append(theta.copy())
    return np.array(losses), np.array(path)

momentum_losses, momentum_path = scaled_momentum_run()
rmsprop_losses, rmsprop_path = scaled_rmsprop_run()
for name, losses in [("Momentum", momentum_losses), ("RMSProp", rmsprop_losses)]:
    reached = np.flatnonzero(losses < 0.1)
    first = int(reached[0]) if len(reached) else "not reached"
    print(f"{name}: loss after 100 updates = {losses[-1]:.6g}; first below 0.1 = {first}")
u_grid = np.linspace(-1.05, 1.05, 240)
v_grid = np.linspace(-1.05, 1.05, 240)
U, V = np.meshgrid(u_grid, v_grid)
Z = 0.5 * (U**2 + 100 * V**2)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].contour(U, V, Z, levels=[.05, .2, .5, 1, 2, 4, 8, 12, 20, 30, 40, 50], colors="0.75")
axes[0].plot(momentum_path[:, 0], momentum_path[:, 1], '.-', markersize=3, label="Momentum")
axes[0].plot(rmsprop_path[:, 0], rmsprop_path[:, 1], '.-', markersize=3, label="RMSProp")
axes[0].plot(0, 0, 'k*', markersize=10)
axes[0].set(xlabel="u", ylabel="v", xlim=(-1.05, 1.05), ylim=(-1.05, 1.05),
            title="Parameter paths (fixed viewing window)")
axes[0].legend(fontsize=8)
axes[1].semilogy(np.arange(len(momentum_losses)), np.maximum(momentum_losses, 1e-16),
                 label="Momentum: lr=0.0009, beta=0.9")
axes[1].semilogy(np.arange(len(rmsprop_losses)), np.maximum(rmsprop_losses, 1e-16),
                 label="RMSProp: lr=0.03, rho=0.99")
axes[1].set(xlabel="Updates", ylabel="Loss", title="Loss includes the initial value")
axes[1].legend(fontsize=8)
fig.tight_layout(); plt.show()

**Your record — 2E:** first RMSProp update / predicted winner and threshold update / explanation.

_Write here._

## Optional 3. How noisy should an update be?

After comparing optimizer history, we now vary how much data contributes to each gradient estimate. This connects optimizer behavior to minibatch noise, a core part of practical training.

We now fit a linear model to **synthetic data**: three informative features and 37 irrelevant features, plus measurement noise. With only 50 training observations, fitting noise is possible even with a linear model. These are not polynomial features and no external dataset is downloaded.

The fitted model is

$$\hat y=b+w_1x_1+\cdots+w_{40}x_{40}.$$

There are **41 fitted parameters**: 40 weights and an intercept. Before standardization, the synthetic targets are generated as $y=1.5x_1-x_2+0.5x_3+\epsilon$, where $\epsilon$ is random noise. The other 37 features have no effect on the generated target, but a model can still fit accidental patterns involving them in a small training sample.

The fixed split has 50 training, 90 validation, and 100 reserved test observations. Feature scaling uses training statistics only. Test data are reserved and never evaluated in this session. The intercept is the first parameter. All reported losses are half MSE; L2, when enabled later, is $\lambda\|w\|^2$ on weights only.

The complete training and plotting code below is supplied. Read the comments identifying the gradient, sampling, and update; you need not implement it. Gradients are calculated explicitly and assigned to PyTorch's SGD optimizer, so using it requires no backpropagation knowledge.

In [ ]:
def make_data(seed=31):
    rng = np.random.default_rng(seed)
    raw = rng.normal(size=(240, 40))
    truth = np.zeros(40)
    truth[:3] = [1.5, -1., .5]
    target = raw @ truth + rng.normal(size=240)
    mu, scale = raw[:50].mean(axis=0), raw[:50].std(axis=0)
    features = (raw - mu) / scale
    design = np.column_stack([np.ones(240), features])
    tensor = lambda a: torch.tensor(a, dtype=DTYPE)
    return (tensor(design[:50]), tensor(target[:50]),
            tensor(design[50:140]), tensor(target[50:140]))

X_train, y_train, X_valid, y_valid = make_data()
print("Training:", tuple(X_train.shape), "Validation:", tuple(X_valid.shape),
      "100 test observations reserved, not exposed to the training experiment.")

def fit(lr=.05, epochs=20, batch_size=50, seed=0, lam=0.):
    n = len(y_train)
    theta = torch.zeros(X_train.shape[1], dtype=DTYPE, requires_grad=True)
    optimizer = torch.optim.SGD([theta], lr=lr)
    rng = np.random.default_rng(seed)
    train_loss, valid_loss, weights = [], [], []
    def record():
        with torch.no_grad():
            train_loss.append(float(.5 * ((X_train @ theta - y_train)**2).mean()))
            valid_loss.append(float(.5 * ((X_valid @ theta - y_valid)**2).mean()))
            weights.append(theta.detach().numpy().copy())
    record()
    updates = 0
    for _ in range(epochs):
        order = rng.permutation(n)  # A fresh shuffle each epoch; no replacement within it.
        for start in range(0, n, batch_size):
            ids = order[start:start + batch_size]  # Retain the final partial batch.
            xb, yb = X_train[ids], y_train[ids]
            with torch.no_grad():
                penalty_grad = 2 * lam * theta.detach().clone()
                penalty_grad[0] = 0.  # Intercept is unpenalized.
                grad = xb.T @ (xb @ theta - yb) / len(ids) + penalty_grad
            optimizer.zero_grad()
            theta.grad = grad
            optimizer.step()  # theta <- theta - lr * grad
            updates += 1
            record()  # Evaluate FULL train/validation losses after every update.
    return {"train": np.array(train_loss), "valid": np.array(valid_loss),
            "weights": np.array(weights), "updates": updates,
            "updates_per_epoch": int(np.ceil(n / batch_size))}


Predict which batch size will add the most variability. Compare the full-training-loss curves across two shuffle seeds.

1. Report updates per epoch and total updates. What does equal epochs control, and what does it leave different?
2. State a conclusion supported across seeds. Why can’t the comparison attribute every difference to gradient noise or establish a universally best batch size?

In [ ]:
# CHANGE THESE CONTROLS: first compare batches, then change only shuffle_seed.
batch_sizes = [1, 10, 50]
shuffle_seed = 0
batch_epochs = 12
batch_runs = {b: fit(lr=.02, epochs=batch_epochs, batch_size=b, seed=shuffle_seed)
              for b in batch_sizes}
fig, ax = plt.subplots(figsize=(8, 4))
for b, run in batch_runs.items():
    epoch_axis = np.arange(len(run["train"])) / run["updates_per_epoch"]
    ax.plot(epoch_axis, run["train"], label=f"batch {b}")
    print(f"Batch {b}: {run['updates_per_epoch']} updates/epoch, {run['updates']} updates total")
ax.set(xlabel="Epochs (fractional positions mark updates)", ylabel="Full training half MSE")
ax.legend(); fig.tight_layout(); plt.show()


**Your record — optional 3:** prediction / results for at least two seeds / update counts / fair-comparison limitation.

_Write here._

## Optional 4. A better fit—or a better model?

Now shift from how updates are taken to whether the fitted model generalizes. Use the same dataset, with **full batches and fixed learning rate 0.05**. Inspect the supplied short-versus-long unregularized comparison, then compare three L2 strengths. This changes the fitting objective, not just the path to the same optimum.

1. Compare training and validation losses at updates 20 and 600, then compare final train/validation losses and weight norms for the three $\lambda$ values. Does lower training loss mean better validation performance? Why use the same unpenalized validation metric?
2. Choose a validation-selected $(\lambda,\text{update})$ pair. Explain why its validation score is not an independent final assessment and how to use the reserved test set.

No guarantee is made that positive L2 beats every early unregularized checkpoint on this particular split.

In [ ]:
# Three guided settings. Additional tuning is optional, after the core discussion.
strengths = [0., .03, .3]
regularized_runs = {lam: fit(lr=.05, epochs=600, batch_size=50, lam=lam)
                    for lam in strengths}
base = regularized_runs[0.]
for t in [20, 600]:
    print(f"Unregularized update {t}: train={base['train'][t]:.4f}, validation={base['valid'][t]:.4f}")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for lam, run in regularized_runs.items():
    best = int(np.argmin(run["valid"]))
    axes[0].plot(run["train"], label=f"lambda={lam}")
    axes[1].plot(run["valid"], label=f"lambda={lam}")
    axes[1].scatter(best, run["valid"][best], s=25)
    norm = np.linalg.norm(run["weights"][-1, 1:])
    print(f"lambda={lam}: final train={run['train'][-1]:.4f}, final val={run['valid'][-1]:.4f}, "
          f"final weight norm={norm:.3f}, best checkpoint={best}, best val={run['valid'][best]:.4f}")
for ax, title in zip(axes, ["Training", "Validation (dots mark selected checkpoints)"]):
    ax.set(title=title, xlabel="Updates", ylabel="Half MSE"); ax.legend()
fig.tight_layout(); plt.show()

**Your record — optional 4:** train/validation comparisons / selected pair / why a final test assessment is still needed.

_Write here._

Return to **worksheet activity 5** for the final exercise.

### Sources
Original classroom tasks aligned with HealthML deep-learning-book Chapter 3. Data and landscapes are synthetic. Half MSE and L2 conventions match that chapter. Repository license: code MIT; prose and exercises CC BY-SA 4.0.